In [ ]:
# repo root + config (walk parents; do not use ../..)
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# MedMentions generative inference (5 generative models)

Run generative models over the **existing** MedMentions validated perturbation set
(do **not** regenerate perturbations).

| Kind | Models |
|---|---|
| Causal instruct | BioMistral-7B, Mistral-7B-Instruct-v0.1, Llama3-OpenBioLLM-8B, Meta-Llama-3-8B-Instruct |
| Seq2seq | FLAN-T5-base (no chat template: CADEC seq2seq path) |

**Input:** live `rq1_sampled_instances.csv` + `rq1_validated_perturbations.csv` (step 2).
Do **not** copy the 550-row archive. Variants keyed by `instance_id` + `input_variant_id`.

**Output:** sharded `outputs/rq1/intermediate/mm_shards/gen_<key>_shardNNNN.csv`
+ sidecars; assembler writes `rq1_generative_model_outputs.csv` and partial
`rq1_all_model_outputs.csv` from instance-blocks complete for all 8 models.
Env: `MM_MODEL_KEY`, `MM_SHARD_SIZE` (default 8000), `MM_MAX_SECONDS`.


In [ ]:
# Setup: absolute PROJECT_ROOT (nbconvert-safe); CUDA required
import json
import os
import shutil
import sys
import time
import gc
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer

PROJECT_ROOT = PROJECT_ROOT
CONFIG_PATH = PROJECT_ROOT / "config" / "config.json"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CONFIG_PATH:  {CONFIG_PATH}")
assert CONFIG_PATH.is_file(), f"Missing config.json: {CONFIG_PATH}"
assert torch.cuda.is_available(), "CUDA required — CPU placement is not allowed for this notebook"

with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)


def _expand_tree(obj):
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _model_src(key: str) -> str:
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def assert_model_on_cuda(model, name: str):
    """Crash loudly if the model landed on CPU."""
    devices = {p.device for p in model.parameters()}
    cuda_devs = {d for d in devices if d.type == "cuda"}
    assert cuda_devs, (
        f"CPU PLACEMENT BUG: {name} has no parameters on CUDA. "
        f"Devices seen: {sorted(str(d) for d in devices)}. "
        f"Refuse to run silently on CPU."
    )
    first = next(model.parameters()).device
    if len(devices) == 1:
        assert first.type == "cuda", (
            f"CPU PLACEMENT BUG: {name} first parameter on {first}, expected cuda"
        )
    print(f"DEVICE OK [{name}]: param_devices={sorted(str(d) for d in devices)} | GPU={torch.cuda.get_device_name(0)}")


def load_generative(key: str):
    """Causal 7–8B — bf16 + device_map=auto (same as CADEC_inference)."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    assert_model_on_cuda(model, f"causal:{key}")
    return tokenizer, model


def load_seq2seq(key: str):
    """FLAN-T5 — NO device_map; explicit .to('cuda') (same as CADEC_inference)."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
    )
    model = model.to("cuda").eval()
    assert_model_on_cuda(model, f"seq2seq:{key}")
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


print(f"GPU: {torch.cuda.get_device_name(0)}")
sys.stdout.flush()


## 1) Copy archive inputs + build MedMentions variant table

Uses the existing encoder `rq1_model_outputs.csv` so `input_text` / variant IDs match
the MedMentions encoder run exactly. Does **not** regenerate perturbations.

In [ ]:
# Live variant table + shard index (no archive copy)
def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()

import sys as _sys
_sys.path.insert(0, str(PROJECT_ROOT))
from scripts.mm_shard_lib import (
    assert_cuda, shard_root, write_instance_index, shard_instance_ids,
    shard_csv, shard_is_complete, write_complete_sidecar, record_shard_model,
    expected_variant_rows, env_model_key, env_shard_id, n_shards,
    should_stop_before_next_shard, GEN_KEYS,
)
from scripts.mm_assemble import assemble_partial_grid, TARGET_COLS

assert_cuda()
INTER_DIR = PROJECT_ROOT / "outputs" / "rq1" / "intermediate"
INTER_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = INTER_DIR / "mm_model_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = INTER_DIR / "rq1_generative_model_outputs.csv"
DST_VALIDATED = INTER_DIR / "rq1_validated_perturbations.csv"
DST_SAMPLED = INTER_DIR / "rq1_sampled_instances.csv"
DST_ENCODER_OUT = INTER_DIR / "rq1_model_outputs.csv"

assert DST_SAMPLED.is_file(), f"Missing {DST_SAMPLED} — run RQ1 sampling/perts first"
assert DST_VALIDATED.is_file(), f"Missing {DST_VALIDATED}"

CAUSAL_MODELS = [
    {"key": "biomistral", "model_name": "BioMistral-7B", "domain": "biomedical"},
    {"key": "mistral", "model_name": "Mistral-7B-Instruct-v0.1", "domain": "general"},
    {"key": "openbiollm", "model_name": "Llama3-OpenBioLLM-8B", "domain": "biomedical"},
    {"key": "llama3", "model_name": "Meta-Llama-3-8B-Instruct", "domain": "general"},
]
SEQ2SEQ_MODELS = [
    {"key": "flan-t5-base", "model_name": "FLAN-T5-base", "domain": "general"},
]
ALL_GENERATIVE = CAUSAL_MODELS + SEQ2SEQ_MODELS
MATCHED_PAIRS = [
    ("BioMistral-7B", "Mistral-7B-Instruct-v0.1"),
    ("Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct"),
]
KEY_TO_SPEC = {s["key"]: s for s in ALL_GENERATIVE}

print("Models to run (generative only):")
for s in ALL_GENERATIVE:
    kind = "causal" if s in CAUSAL_MODELS else "seq2seq"
    print(f"  {kind:7s} {s['model_name']:28s} key={s['key']} src={_model_src(s['key'])}")

df_inst = pd.read_csv(DST_SAMPLED, low_memory=False)
df_pert = pd.read_csv(DST_VALIDATED, low_memory=False)
if "accepted_final" in df_pert.columns:
    df_pert = df_pert[df_pert["accepted_final"] == True].copy()

orig_rows = []
for _, r in df_inst.iterrows():
    orig_rows.append({
        "instance_id": r["instance_id"],
        "input_variant_id": f"{r['instance_id']}_orig",
        "input_type": "original",
        "input_text": r["mention_context"],
        "gold_cui_or_entity": r.get("gold_cui", None),
        "perturbation_type": "original",
    })
pid_col = "perturbation_id" if "perturbation_id" in df_pert.columns else "input_variant_id"
text_col = "perturbation_text" if "perturbation_text" in df_pert.columns else "input_text"
pert_rows = []
for _, r in df_pert.iterrows():
    pert_rows.append({
        "instance_id": r["instance_id"],
        "input_variant_id": r.get(pid_col),
        "input_type": "perturbation",
        "input_text": r.get(text_col),
        "gold_cui_or_entity": r.get("gold_cui", None),
        "perturbation_type": r.get("perturbation_type"),
    })
df_variants = pd.DataFrame(orig_rows + pert_rows)
df_variants["instance_id"] = df_variants["instance_id"].astype(str)
_m = df_variants.groupby("instance_id").size()
keep_ids = set(_m[_m >= 3].index)
n_before = len(df_variants)
df_variants = df_variants[df_variants["instance_id"].isin(keep_ids)].reset_index(drop=True)
_log(
    f"Variant table: {len(df_variants):,} rows (from {n_before:,}) | "
    f"instances={df_variants['instance_id'].nunique():,} | "
    f"m>=3 kept={len(keep_ids):,} excluded={(_m < 3).sum()}"
)
print(df_variants["input_type"].value_counts().to_string())
assert len(df_variants) > 0

MM_ROOT = shard_root(PROJECT_ROOT)
# Residual order = sampled CSV order (seed 42 recorded; no downsample)
write_instance_index(MM_ROOT, df_inst["instance_id"].astype(str).tolist(), seed=42)
_n_sh = n_shards(len(df_inst))
_log(f"Shard index: n_instances={len(df_inst):,} n_shards={_n_sh} dir={MM_ROOT}")
sys.stdout.flush()


## 2) Inference helpers (causal chat-template + FLAN seq2seq from CADEC_inference)

In [ ]:
# Causal generative inference (CADEC_inference generate_concept)
CAUSAL_MAX_NEW_TOKENS = 16  # short concept span, not prose
_CONCEPT_INSTR = (
    "Identify the primary medical concept in the following clinical text. "
    "Reply with only the concept name.\n\n"
    "Text: {text}"
)


def raw_path_for(key: str, shard_id: int | None = None) -> Path:
    if shard_id is None:
        return RAW_DIR / f"mm_raw_{key}.csv"
    return shard_csv(MM_ROOT, "gen", key, shard_id)


# Llama-3 chat template (OpenBioLLM ships without one)
_LLAMA3_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
    "{{ content }}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)


def _ensure_chat_template(tokenizer) -> None:
    """Attach a chat template when missing (OpenBioLLM = Llama-3 family)."""
    if getattr(tokenizer, "chat_template", None):
        return
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk:
        tokenizer.chat_template = _LLAMA3_CHAT_TEMPLATE
        return
    tokenizer.chat_template = (
        "{{ bos_token }}{% for message in messages %}"
        "{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}{{ message['content'] }}"
        "{% endif %}{% endfor %}"
    )


def _clean_concept_output(decoded: str) -> str:
    """Strip residual chat-template / scaffolding markers from decoded span."""
    text = decoded.strip()
    for marker in ("[/INST]", "</s>", "<s>"):
        text = text.replace(marker, " ")
    for marker in ("Answer:", "Concept:", "The primary medical concept is"):
        if marker in text:
            text = text.split(marker)[-1]
    m = re.search(
        r"(?is)the primary medical concepts?\b.*?\b(?:is|are)\b\s*:?\s*",
        text,
    )
    if m:
        text = text[m.end():]
    text = text.strip(" \"'`.")
    text = " ".join(text.split()).strip()
    return text[:200]


def generate_concept(text: str, tokenizer, model) -> str:
    _ensure_chat_template(tokenizer)
    user_content = _CONCEPT_INSTR.format(text=text)
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=True)
    try:
        first_dev = next(model.parameters()).device
        enc = {k: v.to(first_dev) for k, v in enc.items()}
    except StopIteration:
        enc = {k: v.to("cuda:0") for k, v in enc.items()}

    input_len = enc["input_ids"].shape[-1]
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk and eot not in eos_ids:
        eos_ids.append(eot)
    gen_kwargs = dict(
        max_new_tokens=CAUSAL_MAX_NEW_TOKENS,
        do_sample=False,  # greedy, T=0
        pad_token_id=tokenizer.pad_token_id,
    )
    if eos_ids:
        gen_kwargs["eos_token_id"] = eos_ids if len(eos_ids) > 1 else eos_ids[0]
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    new_tokens = out[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return _clean_concept_output(decoded)


def _rows_from_generations(spec, gens, block):
    rows = []
    for (_, row), gen in zip(block.iterrows(), gens):
        rows.append({
            "instance_id": row["instance_id"],
            "model_name": spec["model_name"],
            "input_variant_id": row["input_variant_id"],
            "input_type": row["input_type"],
            "output_text": gen,
            "gold_cui_or_entity": row["gold_cui_or_entity"],
            "perturbation_type": row["perturbation_type"],
        })
    return rows


def run_one_causal(spec: dict, block: pd.DataFrame, out_path: Path) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    src = _model_src(key)
    _log(f"LOAD causal {model_name} from {src} (bf16, device_map=auto, T=0 greedy) n={len(block):,}")
    t0 = time.perf_counter()
    tokenizer, model = load_generative(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    _ensure_chat_template(tokenizer)

    # EXECUTION-ONLY speedup: batched left-padded generation when MM_CAUSAL_BATCH>1.
    # Default (unset/1) = the exact single-sequence path below (unchanged). Batching is
    # enabled per-model ONLY after scripts/mm_gen_verify.py proves it byte-identical.
    _causal_bs = int(os.environ.get("MM_CAUSAL_BATCH", "1"))
    if _causal_bs > 1:
        import scripts.mm_gen_batch as _gbz
        _texts = [str(t) if pd.notna(t) else "" for t in block["input_text"].tolist()]
        _log(f"  BATCHED causal generate: n={len(_texts):,} batch_size={_causal_bs} (verified byte-identical)")
        gens = _gbz.generate_concept_batch(_texts, tokenizer, model, batch_size=_causal_bs)
    else:
        gens = []
        for i, row in tqdm(block.iterrows(), total=len(block), desc=model_name):
            text = str(row["input_text"]) if pd.notna(row["input_text"]) else ""
            try:
                gens.append(generate_concept(text, tokenizer, model))
            except Exception as e:
                _log(f"WARN generate failed {model_name} row={i}: {e}")
                gens.append("")
            if (len(gens) % 500) == 0:
                _log(f"  {model_name}: {len(gens)}/{len(block)} elapsed={time.perf_counter()-t0:.0f}s")

    df_out = pd.DataFrame(_rows_from_generations(spec, gens, block))
    df_out.to_csv(out_path, index=False)
    n_unique = df_out["output_text"].fillna("").astype(str).nunique()
    _log(
        f"DONE {model_name}: rows={len(df_out)} unique_outputs={n_unique} "
        f"mean_len={df_out['output_text'].astype(str).str.len().mean():.1f} -> {out_path.name}"
    )
    if n_unique < 200:
        _log(f"WARN degeneracy check: {model_name} unique_outputs={n_unique} (want thousands, not ~140)")

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


# FLAN-T5 seq2seq (CADEC_inference: no chat template)
MAX_NEW_TOKENS = 32  # same as CADEC FLAN path


def generate_concept_seq2seq_batch(texts, tokenizer, model, batch_size=32):
    """Batched FLAN-T5 greedy decode on CUDA."""
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) if t is not None else "" for t in texts[i:i + batch_size]]
        prompts = [
            "Identify the primary medical concept in the following clinical text. "
            "Reply with only the concept name.\n\n"
            f"Text: {t}"
            for t in batch
        ]
        enc = tokenizer(
            prompts, return_tensors="pt", truncation=True, max_length=512, padding=True
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        outs.extend([d.strip()[:200] for d in decoded])
    return outs


def run_one_seq2seq(spec: dict, block: pd.DataFrame, out_path: Path) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    src = _model_src(key)
    _log(f"LOAD seq2seq {model_name} from {src} (bf16, explicit cuda, T=0 greedy) n={len(block):,}")
    t0 = time.perf_counter()
    tokenizer, model = load_seq2seq(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    texts = block["input_text"].fillna("").astype(str).tolist()
    _log(f"  Batched seq2seq generate on cuda, n={len(texts):,} batch_size=32")
    gens = generate_concept_seq2seq_batch(texts, tokenizer, model, batch_size=32)

    df_out = pd.DataFrame(_rows_from_generations(spec, gens, block))
    df_out.to_csv(out_path, index=False)
    n_unique = df_out["output_text"].fillna("").astype(str).nunique()
    _log(
        f"DONE {model_name}: rows={len(df_out)} unique_outputs={n_unique} "
        f"mean_len={df_out['output_text'].astype(str).str.len().mean():.1f} -> {out_path.name}"
    )
    if n_unique < 200:
        _log(f"WARN degeneracy check: {model_name} unique_outputs={n_unique}")

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


## 3) Run generative models (sharded; resume via complete sidecars)


In [ ]:
# Sharded generative run (one model per SLURM array task)
import time as _time
_t0 = _time.time()
os.environ["MM_JOB_T0"] = str(_t0)
assert torch.cuda.is_available(), "CUDA required at job start"

_want = env_model_key()
if _want:
    specs = [KEY_TO_SPEC[_want]] if _want in KEY_TO_SPEC else None
    if specs is None:
        raise KeyError(f"MM_MODEL_KEY={_want!r} not in {list(KEY_TO_SPEC)}")
else:
    specs = list(ALL_GENERATIVE)
    _log("MM_MODEL_KEY unset — running ALL generative models (not the array path)")

_only = env_shard_id()
_n_inst = len(pd.read_csv(DST_SAMPLED, usecols=["instance_id"]))
_n_sh = n_shards(_n_inst)
_shards = [_only] if _only is not None else list(range(_n_sh))
_log(f"Generative job models={[s['key'] for s in specs]} shards={_shards}")

for spec in specs:
    for sid in _shards:
        if should_stop_before_next_shard(_t0):
            _log(f"WALLTIME checkpoint before {spec['key']} shard {sid} — stop")
            assemble_partial_grid(PROJECT_ROOT, require_all_eight=True)
            raise SystemExit(0)
        ids = shard_instance_ids(MM_ROOT, sid)
        block = df_variants[df_variants["instance_id"].isin(ids)].copy()
        exp = expected_variant_rows(df_variants, ids)
        out_p = shard_csv(MM_ROOT, "gen", spec["key"], sid)
        if shard_is_complete(out_p, expected_rows=exp):
            _log(f"SKIP complete gen {spec['key']} shard {sid} rows={exp}")
            continue
        if exp == 0:
            pd.DataFrame(columns=TARGET_COLS).to_csv(out_p, index=False)
            meta = write_complete_sidecar(out_p, {"n_instances": 0, "expected_rows": 0})
            record_shard_model(MM_ROOT, "gen", spec["key"], sid, meta)
            _log(f"COMPLETE gen {spec['key']} shard {sid} (empty)")
            continue
        _log(f"RUN gen {spec['key']} shard {sid} variants={len(block):,} expected={exp}")
        runner = run_one_seq2seq if spec in SEQ2SEQ_MODELS else run_one_causal
        runner(spec, block, out_p)
        assert out_p.is_file()
        n = sum(1 for _ in open(out_p, encoding="utf-8", errors="replace")) - 1
        assert n == exp, f"row-count {n} != expected {exp} for {out_p}"
        meta = write_complete_sidecar(out_p, {"n_instances": len(ids), "expected_rows": exp})
        record_shard_model(MM_ROOT, "gen", spec["key"], sid, meta)
        _log(f"COMPLETE gen {spec['key']} shard {sid}")

_log("Generative shard loop finished")
assemble_partial_grid(PROJECT_ROOT, require_all_eight=True)


## 4) Assemble generative CSV + `rq1_all_model_outputs.csv` (8 models)

In [ ]:
# Assemble from complete shards (partial grid OK)
from scripts.mm_assemble import assemble_partial_grid, TARGET_COLS
from scripts.mm_shard_lib import grid_status, complete_shards_for_grid, shard_csv, shard_is_complete, GEN_KEYS

st = grid_status(MM_ROOT)
print("manifest/grid:", {k: st[k] for k in ("n_instances","n_shards","complete_grid_shards")})
info = assemble_partial_grid(PROJECT_ROOT, require_all_eight=True)
print("assembled", info)

ready = complete_shards_for_grid(MM_ROOT)
if not ready:
    _log("No 8-model-complete instance-block yet — skip degeneracy/pair asserts")
else:
    # degeneracy on concatenated complete gen shards (not a tiny first shard alone)
    parts=[]
    for sid in ready:
        for spec in ALL_GENERATIVE:
            pp = shard_csv(MM_ROOT, "gen", spec["key"], sid)
            df_p = pd.read_csv(pp)
            parts.append((spec["model_name"], df_p))
    by_model = {}
    for name, df_p in parts:
        by_model.setdefault(name, []).append(df_p)
    print("===== Per-model unique-output degeneracy (complete blocks only) =====")
    for spec in ALL_GENERATIVE:
        df_p = pd.concat(by_model[spec["model_name"]], ignore_index=True)
        n_unique = df_p["output_text"].fillna("").astype(str).nunique()
        empty_frac = (df_p["output_text"].fillna("").astype(str).str.strip()=="").mean()
        print(f"{spec['model_name']:28s} rows={len(df_p):7d} unique={n_unique:6d} empty={empty_frac:.3f}")
        if len(df_p) >= 2000:
            assert n_unique >= 200, f"DEGENERACY {spec['model_name']} unique={n_unique}"
ALL_OUT = INTER_DIR / "rq1_all_model_outputs.csv"
_log(f"ALL_OUT exists={ALL_OUT.is_file()} size={ALL_OUT.stat().st_size if ALL_OUT.is_file() else 0}")
